# 05 — Batch inference via the UC alias

Loads `models:/{catalog}.{schema}.xg_model@champion` and scores a fresh batch of
shots. Writes results to `shot_predictions` so we can use it as the source table
for the Lakehouse Monitor in `07`.

**The key point of the demo here:** the code below has **no run_id, no version
number, and no model file path** — only the alias. When you promote a new
version to `@champion` later, this exact code picks up the new model with no
edits. That's the lifecycle story.

In [1]:
import os
import pandas as pd
import mlflow
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "hockey_xg_mlflow")
MODEL_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.xg_model"
FEATURES   = f"{UC_CATALOG}.{UC_SCHEMA}.shots_features"
PRED_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.shot_predictions"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
print(f"Loading {MODEL_NAME}@champion")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Loading alexander_booth.hockey_xg_mlflow.xg_model@champion


In [2]:
import mlflow.pyfunc
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print(f"Loaded. Metadata signature:\n{model.metadata.signature}")

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded. Metadata signature:
inputs: 
  ['distance_ft': double (required), 'angle_deg': double (required), 'distance_sq': double (required), 'angle_sq': double (required), 'is_high_danger': double (required), 'rebound': double (required), 'rush': double (required), 'rebound_dist': double (required), 'rush_dist': double (required), 'period': double (required), 'time_in_period_sec': double (required), 'prev_event_sec': double (required), 'st_wrist': double (required), 'st_snap': double (required), 'st_slap': double (required), 'st_backhand': double (required), 'st_tip': double (required), 'st_wrap': double (required), 'str_5v5': double (required), 'str_5v4_pp': double (required), 'str_4v5_pk': double (required), 'str_4v4': double (required), 'str_3v3_ot': double (required), 'str_6v5_en': double (required), 'str_5v6': double (required)]
outputs: 
  [Tensor('float64', (-1,))]
params: 
  None



## Score a fresh batch

We pull the full feature table and score it. In production this would be the
day's new shot events landing in bronze/silver.

In [3]:
FEATURE_COLS = [
    "distance_ft", "angle_deg", "distance_sq", "angle_sq",
    "is_high_danger", "rebound", "rush", "rebound_dist", "rush_dist",
    "period", "time_in_period_sec", "prev_event_sec",
    "st_wrist", "st_snap", "st_slap", "st_backhand", "st_tip", "st_wrap",
    "str_5v5", "str_5v4_pp", "str_4v5_pk", "str_4v4", "str_3v3_ot", "str_6v5_en", "str_5v6",
]
META_COLS = ["shot_id", "game_id", "team_id", "opp_team_id", "player_id",
             "period", "shot_type", "strength_state", "goal"]

pdf = spark.table(FEATURES).select(*META_COLS, *[c for c in FEATURE_COLS if c not in META_COLS]).toPandas()
print(f"Scoring {len(pdf):,} shots")

X = pdf[FEATURE_COLS].astype("float64")
# Most sklearn classifiers via pyfunc return P(class=1) as a 1-D array
proba = model.predict(X)
# Defensive: some flavors return a DataFrame
if hasattr(proba, "values"):
    proba = proba.values.reshape(-1)
pdf["xg"] = proba
print(pdf[["shot_id", "shot_type", "strength_state", "goal", "xg"]].head(10))

Scoring 50,000 shots
   shot_id shot_type strength_state  goal        xg
0        1     wrist            5v5     0  0.034695
1        2       tip            5v5     0  0.112438
2        3     wrist            5v5     0  0.031859
3        4     wrist            5v5     0  0.111754
4        5  backhand            5v5     0  0.017254
5        6     wrist            5v5     0  0.107582
6        7       tip            5v6     0  0.108935
7        8     wrist         6v5_en     1  0.596917
8        9     wrist            5v5     0  0.033328
9       10     wrist            5v5     0  0.113172


## Write predictions table

We add the model URI as a column so downstream consumers (or the monitor) know
which version produced each row. When `@champion` flips to a new version, the
next batch's `model_uri` reflects that automatically.

In [4]:
from datetime import datetime, timezone, timedelta
from mlflow.tracking import MlflowClient

client = MlflowClient()
mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
model_uri_str = f"models:/{MODEL_NAME}/{mv.version}"

pdf["prediction"]     = (pdf["xg"] >= 0.5).astype(int)
pdf["model_version"]  = int(mv.version)
pdf["model_uri"]      = model_uri_str
# Backdate by 1 day so the drift batch in 07 lands in a separate daily monitoring window
pdf["scored_at_utc"]  = datetime.now(timezone.utc) - timedelta(days=1)
pdf["batch_id"]       = "initial_backfill"

sdf = spark.createDataFrame(pdf)
(sdf.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PRED_TABLE))
print(f"Wrote {sdf.count():,} rows to {PRED_TABLE} (model v{mv.version})")
spark.sql(f"SELECT shot_id, shot_type, strength_state, goal, xg, model_version FROM {PRED_TABLE} ORDER BY xg DESC LIMIT 5").show(truncate=False)

Wrote 50,000 rows to alexander_booth.hockey_xg_mlflow.shot_predictions (model v5)


+-------+---------+--------------+----+------------------+-------------+
|shot_id|shot_type|strength_state|goal|xg                |model_version|
+-------+---------+--------------+----+------------------+-------------+
|31111  |tip      |6v5_en        |1   |0.9242914902316602|5            |
|23450  |tip      |6v5_en        |1   |0.9103775158112463|5            |
|43624  |tip      |6v5_en        |1   |0.8712363456597465|5            |
|31052  |tip      |6v5_en        |1   |0.8492716543294013|5            |
|15444  |snap     |6v5_en        |1   |0.8471176030298562|5            |
+-------+---------+--------------+----+------------------+-------------+



## Lineage check

UC tracks lineage from the model version back to the run, and from the
predictions table back to the upstream feature table. You can see this in the
Catalog Explorer UI under both the model version and the predictions table.

In [5]:
print(f"View model lineage in UI:")
print(f"  https://e2-demo-field-eng.cloud.databricks.com/explore/data/models/{UC_CATALOG}/{UC_SCHEMA}/xg_model")
print(f"\nView table lineage in UI:")
print(f"  https://e2-demo-field-eng.cloud.databricks.com/explore/data/{UC_CATALOG}/{UC_SCHEMA}/shot_predictions")

View model lineage in UI:
  https://e2-demo-field-eng.cloud.databricks.com/explore/data/models/alexander_booth/hockey_xg_mlflow/xg_model

View table lineage in UI:
  https://e2-demo-field-eng.cloud.databricks.com/explore/data/alexander_booth/hockey_xg_mlflow/shot_predictions


**Next:** `06_serving_endpoint` deploys this same `@champion` to a low-latency
serving endpoint so external apps can score one shot at a time.